# Create Novo Nordisk Fonden Prize Recipients (PRIZE PATTERN, method-5 static HTML)

Prize recipients from the foundation's WordPress archive at novonordiskfonden.dk. Novo Nordisk Fonden (Danish: "Novo Nordisk Foundation") is one of the world's largest biomedical-research funders by endowment (DKK 100+ billion). The foundation awards multiple named prizes (Novo Nordisk Prisen, Marie og August Krogh Prisen, Hagedorn Prisen, Nordic Diabetes Prize, etc.).

**Prerequisites:**
- Run `scripts/local/novo_nordisk_fonden_to_s3.py` first.

**Data source:** `/prize_recipients-sitemap.xml` enumerates **~381 recipient URLs** at `/prismodtagere/{name-slug}-{year}/`. Each page exposes:
- `<h1>` recipient name
- `<h4>` credentials (e.g., "Professor, dr.med.")
- Header strip: "Prize Name - YEAR"

**S3 location:** `s3a://openalex-ingest/awards/novo_nordisk_fonden/novo_nordisk_fonden_recipients.parquet`

**Awarding body:**
- funder_id: 4320325957
- display_name: "Novo Nordisk Fonden"
- country: DK
- ROR: none
- DOI: 10.13039/501100009708

**Coverage (smoke test 10 recipients, 2026-05-23):**
- 100% recipient_name / slug / credentials / prize_name / award_year

**Amount/currency:** NULL with §6.7 waiver. Foundation publishes per-prize amount info on `/priser/{prize-slug}/` pages (e.g., Novo Nordisk Prisen carries DKK 5M) but NOT on recipient pages. Prize-pattern precedent: Fields Medal #50, Royal Society Medals #71, Wolf Prize #47, Lasker #48.

**Provenance:** `novo_nordisk_fonden_prizes`.

**Priority:** 119.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.novo_nordisk_fonden_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/novo_nordisk_fonden/novo_nordisk_fonden_recipients.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.novo_nordisk_fonden_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.novo_nordisk_fonden_raw;


## Step 1.5: Inspect Raw Data, Money Scan, Native Key

In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'novo_nordisk_fonden_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'novo_nordisk_fonden_raw'
  AND LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT recipient_name, slug, credentials, prize_name, award_year
FROM openalex.awards.novo_nordisk_fonden_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.novo_nordisk_fonden_raw;


In [ ]:
%sql
SELECT prize_name, COUNT(*) as recipients
FROM openalex.awards.novo_nordisk_fonden_raw
GROUP BY prize_name ORDER BY recipients DESC LIMIT 20;


In [ ]:
%sql
SELECT award_year, COUNT(*) as recipients
FROM openalex.awards.novo_nordisk_fonden_raw
WHERE award_year IS NOT NULL
GROUP BY award_year ORDER BY award_year DESC LIMIT 30;


## Step 1.6: Fail-fast — Verify the NNF Funder Row Exists

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320325957;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.novo_nordisk_fonden_awards
USING delta
AS
WITH
nnf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320325957
),
raw_prepared AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS parsed_year,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date
    FROM openalex.awards.novo_nordisk_fonden_raw
    WHERE recipient_name IS NOT NULL
      AND TRIM(recipient_name) != ''
      AND funder_award_id IS NOT NULL
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.recipient_name as display_name,
        g.credentials as description,

        f.funder_id,
        g.funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'prize' as funding_type,
        COALESCE(NULLIF(TRIM(g.prize_name), ''), 'Novo Nordisk Fonden Prize') as funder_scheme,

        'novo_nordisk_fonden_prizes' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        g.parsed_year as start_year,
        CAST(NULL AS INT) as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.landing_page_url as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM raw_prepared g
    CROSS JOIN nnf_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 119

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'novo_nordisk_fonden_prizes' AND priority = 119;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    119 as priority
FROM openalex.awards.novo_nordisk_fonden_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total FROM openalex.awards.novo_nordisk_fonden_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.novo_nordisk_fonden_awards;


In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_pi
FROM openalex.awards.novo_nordisk_fonden_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, funder_scheme,
       lead_investigator.given_name AS pi_given,
       lead_investigator.family_name AS pi_family
FROM openalex.awards.novo_nordisk_fonden_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.novo_nordisk_fonden_awards
GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.novo_nordisk_fonden_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.novo_nordisk_fonden_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 20;


In [ ]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids
FROM openalex.awards.novo_nordisk_fonden_awards;
